In [ ]:
# =======================
# 1. Imports
# =======================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
import optuna
import mlflow
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow.sklearn

In [ ]:
# ============================
# 2. Load datasets
# =============================
train_df = pd.read_csv(r'C:\Users\sheri\Desktop\machinlearning\data\processed_data\feature_engineered_train.csv')
eval_df = pd.read_csv(r'C:\Users\sheri\Desktop\machinlearning\data\processed_data\featured_engineered_eval.csv')

# Define target & features
target = "AEP_MW"
X_train, y_train  = train_df.drop(columns=[target]), train_df[target]

X_eval, y_eval = eval_df.drop(columns=[target]), eval_df[target]

print("Train Shape:", X_train.shape)
print("Eval Shape:", X_eval.shape)

Train Shape: (62920, 4)
Eval Shape: (34923, 4)


In [ ]:
# ======================================================
# 3. Define OPtuna objective function with MLFlow
# ======================================================
def objective(trial):
    params = {
    "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
    "min_weight_fraction_leaf": trial.suggest_int("min_weight_fraction_leaf", 0.0, 0.5),
    "max_depth": trial.suggest_int("max_depth", 3, 10),
    "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 6),
    "random_state": 42,
    "alpha": trial.suggest_float("alpha", 0.0, 1.0),
    "tol": trial.suggest_float("tol", 0.0, 7),
    "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 8)
    }

    with mlflow.start_run(nested=True):
        model = GradientBoostingRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # log hyperparameter + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    return rmse

    

In [ ]:
# =====================================
# 4. Run Optuna study with MLFlow
# =====================================
# Force MLFlow to always use the root project mlrun folder
#mlflow.set_tracking_uri(r'C:\Users\sheri\Desktop\machinlearning\mlruns')
mlflow.set_tracking_uri("file:///C:/Users/sheri/Desktop/machinlearning/mlruns")
mlflow.set_experiment("GradientBoostingRegressor_energy")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best params:", study.best_params)

[I 2026-07-13 09:25:18,023] A new study created in memory with name: no-name-64356619-ad6a-4797-8e0e-ab4c974b755e
[I 2026-07-13 09:25:25,477] Trial 0 finished with value: 1750.9827613225484 and parameters: {'n_estimators': 230, 'learning_rate': 0.011427561569599152, 'subsample': 0.9761240685991417, 'min_samples_split': 2, 'min_samples_leaf': 6, 'min_weight_fraction_leaf': 0, 'max_depth': 6, 'min_impurity_decrease': 1.323562196264263, 'alpha': 0.722795876550425, 'tol': 0.6230610015863741, 'ccp_alpha': 6.199546383216524}. Best is trial 0 with value: 1750.9827613225484.
[I 2026-07-13 09:25:41,499] Trial 1 finished with value: 1923.461420456349 and parameters: {'n_estimators': 357, 'learning_rate': 0.017996189621087212, 'subsample': 0.8976970678650482, 'min_samples_split': 15, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0, 'max_depth': 10, 'min_impurity_decrease': 5.841872743171019, 'alpha': 0.8794655820122379, 'tol': 2.3144359870460525, 'ccp_alpha': 7.235963213183597}. Best is tria

Best params: {'n_estimators': 207, 'learning_rate': 0.014903868584267692, 'subsample': 0.7756663441148834, 'min_samples_split': 4, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0, 'max_depth': 7, 'min_impurity_decrease': 3.8537889347800633, 'alpha': 0.4817462249414025, 'tol': 1.380477225045889, 'ccp_alpha': 2.203529953708492}


In [ ]:
# ============================================================
# 5. Train final model with best params and log to MLFlow
# ============================================================
best_params = study.best_trial.params
best_model = GradientBoostingRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

# log final model
with mlflow.start_run(run_name="best_GradientBoostingRegressor_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.sklearn.log_model(sk_model=best_model, name="model") # or artifact_path="model" for MLflow 2.x

Final tuned model performance:
MAE: 1370.2633497264428
RMSE: 1720.2542036656102
R2: 0.5129058245918845
